## Directory Configuration

Define the main paths used in the project:

- **PROJECT_DIR**: root project directory.
- **DATA_DIR**: location of input data.
- **OUTPUT_DIR**: location where results and generated files will be saved.

The output directory is created automatically if it does not already exist.

In [ ]:
from pathlib import Path
import sys
sys.path.append(str(Path("../code").resolve()))
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs_notebook_sanity"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print("PROJECT_DIR =", PROJECT_DIR)
print("DATA_DIR    =", DATA_DIR)
print("OUTPUT_DIR  =", OUTPUT_DIR)

PROJECT_DIR = /home/alan/adversarial_on_DB/code
DATA_DIR    = /home/alan/adversarial_on_DB/code/data
OUTPUT_DIR  = /home/alan/adversarial_on_DB/code/outputs_notebook_sanity


In [2]:
import os
import time
import math
import json
import copy
import random
import argparse
from datetime import datetime
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple, Literal

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch import Tensor, nn
from torch.nn import Embedding, ModuleDict

from sentence_transformers import SentenceTransformer

from relbench.datasets import get_dataset
from relbench.tasks import get_task
from relbench.modeling.utils import get_stype_proposal
from relbench.modeling.graph import get_node_train_table_input, make_pkey_fkey_graph
from relbench.modeling.nn import HeteroEncoder, HeteroGraphSAGE, HeteroTemporalEncoder

from torch_frame.config.text_embedder import TextEmbedderConfig
from torch_frame.data.stats import StatType

from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import MLP, MessagePassing, HeteroConv
from torch_geometric.typing import NodeType
from torch_geometric.seed import seed_everything

In [ ]:
# ============================================================
# PATHS AND DATASET CONFIGURATION
# ============================================================

root_dir: str = str(DATA_DIR)
output_dir: str = str(OUTPUT_DIR)
cache_dir: str = str(DATA_DIR / "rel-f1_materialized_cache")

dataset_name: str = "rel-f1"
split: str = "val"
device: str = "cpu"


# ============================================================
# TASK CONFIGURATION
# ============================================================

TASK_NAME = "qualifying-position"   
# Options:
# - "driver-dnf"
# - "driver-top3"
# - "driver-position"
# - "qualifying-position"

TASK_NAMES = [TASK_NAME]


if TASK_NAME in ["driver-position", "qualifying-position"]:
    TASK_TYPE = "regression"
    GRADIENT_OBJECTIVE = "mae"
    EVAL_METRIC = "mae"
else:
    TASK_TYPE = "classification"
    GRADIENT_OBJECTIVE = "bce"
    EVAL_METRIC = "accuracy"


# ============================================================
# TRAINING HYPERPARAMETERS
# ============================================================

epochs: int = 1
lr: float = 0.005
batch_size: int = 512

num_layers: int = 2
channels: int = 128
aggr: str = "sum"
norm: str = "batch_norm"
num_neighbors: int = 128


# ============================================================
# ATTACK CONFIGURATION
# ============================================================

fracs: str = "0.001"
budgets = np.linspace(1, 500, 5).astype(int)

topk_per_child_values: str = "1,5,10"
score_norm: str = "raw"

objective_mode: str = "bce"
direction: str = "increase"

q: float = 0.75
alpha: float = 0.25

max_relations: int = 0
allow_dangerous: bool = False


# ============================================================
# SANITY CHECK / DEBUG CONFIGURATION
# ============================================================

SEEDS = [39]  # 39, 40, 41, 42, 43

n_val_batches = 1
n_random = 1

gradient_shortlist_size = 50
random_shortlist_size = 50

max_candidates_per_dst = 100
random_candidates_per_dst = 100

NotebookConfig(root_dir='/home/alan/adversarial_on_DB/code/data', output_dir='/home/alan/adversarial_on_DB/code/outputs_notebook_sanity', cache_dir='/home/alan/adversarial_on_DB/code/data/rel-f1_materialized_cache', dataset_name='rel-f1', task_name='driver-dnf', split='val', device='cpu', seed=42, epochs=1, lr=0.005, batch_size=512, num_layers=2, channels=128, aggr='sum', norm='batch_norm', num_neighbors=128, fracs='0.001', n_random=1, max_candidates_per_dst=50, topk_per_child_values='1,5,10', score_norm='raw', objective_mode='bce', direction='increase', q=0.75, alpha=0.25, max_relations=0, task_type='regression', allow_dangerous=False)


In [ ]:

device = torch.device(device)
log("Loading dataset and task")
dataset = safe_get_dataset(
    dataset_name,
    root_dir=root_dir,
    download=True,
)

task = safe_get_task(
    dataset_name,
    TASK_NAME,
    root_dir=root_dir,
    download=True,
)

task_type = (
    infer_task_type(TASK_NAME, task)
    if TASK_TYPE == "auto"
    else TASK_TYPE
)

entity_table = task.entity_table

if TASK_NAMES != "badges-class" :
    out_channels = 1
else : 
    out_channels = 3

print("task_type:", task_type)
print("entity_table:", entity_table)

# ---------------------------------------------------------------------
# Reload clean task tables
# ---------------------------------------------------------------------

train_table = safe_get_task_table(task, "train")
val_table = safe_get_task_table(task, "val")
test_table = safe_get_task_table(task, "test")

print("train columns:", train_table.df.columns.tolist())
print("val columns:", val_table.df.columns.tolist())
print("test columns:", test_table.df.columns.tolist())

# IMPORTANT:
# Do NOT rename position -> y here.
# RelBench transform will create batch[entity_table].y itself.

if TASK_NAME == "driver-position":
    assert (
        "position" in train_table.df.columns or "y" in train_table.df.columns
    ), train_table.df.columns.tolist()

    assert (
        "position" in val_table.df.columns or "y" in val_table.df.columns
    ), val_table.df.columns.tolist()

# ---------------------------------------------------------------------
# Load DB
# ---------------------------------------------------------------------

log("Loading Database object")
db = dataset.get_db()

# ---------------------------------------------------------------------
# Build graph
# ---------------------------------------------------------------------

log("Building text embedder config")

text_embedder_cfg = TextEmbedderConfig(
    text_embedder=GloveTextEmbedding(device=cfg.device),
    batch_size=256,
)

log("Building stype proposal and graph")

stype_proposal = get_stype_proposal(db)

data, col_stats_dict = make_pkey_fkey_graph(
    db,
    col_to_stype_dict=stype_proposal,
    text_embedder_cfg=text_embedder_cfg,
    cache_dir=cache_dir,
)

data = data.to(device)

# ---------------------------------------------------------------------
# Build loaders like RelBench tutorial
# ---------------------------------------------------------------------

from relbench.modeling.graph import get_node_train_table_input
from torch_geometric.loader import NeighborLoader

loader_dict = {}

for split_name, table in [
    ("train", train_table),
    ("val", val_table),
    ("test", test_table),
]:
    table_input = get_node_train_table_input(
        table=table,
        task=task,
    )

    entity_table = table_input.nodes[0]

    loader_dict[split_name] = NeighborLoader(
        data=data,
        num_neighbors=[num_neighbors for _ in range(num_layers)],
        time_attr="time",
        input_nodes=table_input.nodes,
        input_time=table_input.time,
        transform=table_input.transform,
        batch_size=batch_size,
        temporal_strategy="uniform",
        shuffle=split_name == "train",
        num_workers=0,
        persistent_workers=False,
    )

print("loader_dict keys:", loader_dict.keys())
print("entity_table:", entity_table)

# ---------------------------------------------------------------------
# Sanity check train batch
# ---------------------------------------------------------------------

batch = next(iter(loader_dict["train"])).to(device)

print("train batch keys:", batch[entity_table].keys())

assert "y" in batch[entity_table], (
    f"No y in batch[{entity_table}]. "
    f"Available keys: {batch[entity_table].keys()}"
)

print("batch y shape:", batch[entity_table].y.shape)
print("batch seed_time shape:", batch[entity_table].seed_time.shape)

[2026-06-01 17:02:36] Loading dataset and task
task_type: regression
entity_table: drivers
train columns: ['date', 'driverId', 'did_not_finish']
val columns: ['date', 'driverId', 'did_not_finish']
test columns: ['date', 'driverId']
[2026-06-01 17:02:36] Loading Database object
Loading Database object from /home/alan/.cache/relbench/rel-f1/db...
Done in 0.07 seconds.
[2026-06-01 17:02:36] Building text embedder config


[2026-06-01 17:02:41] Building stype proposal and graph


/home/alan/adversarial_on_DB/code/.adversarial/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/alan/adversarial_on_DB/code/.adversarial/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/alan/adversarial_on_DB/code/.adversarial/lib/python3.12/site-packages/relbench/modeling/graph.py:93: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor 

loader_dict keys: dict_keys(['train', 'val', 'test'])
entity_table: drivers
train batch keys: KeysView({'tf': TensorFrame(
  num_cols=6,
  num_rows=512,
  timestamp (1): ['dob'],
  embedding (5): ['code', 'driverRef', 'forename', 'nationality', 'surname'],
  has_target=False,
  device='cpu',
), 'n_id': tensor([100, 360, 210,  76,  43,  76, 432,  76, 181, 385, 316, 429, 202, 579,
        136,  78,  13, 122, 172, 512, 762, 626, 433, 232, 199, 136, 590,  99,
         75, 132, 590,  54, 484, 537,  76, 363, 277, 242, 181,  55,  48, 306,
         59, 624, 139, 304, 360, 277, 220, 340, 249, 541, 158, 103, 165,  94,
        137, 102, 171, 130, 372,  55,  83,  53, 501, 400, 135,  56,  94, 182,
         55, 232, 198, 340,  48, 130,  22, 641, 180, 128,   3,  83, 642, 242,
         64, 372, 232, 653, 385,  94,  22, 101,  13, 688, 118, 109,  62, 456,
        615, 660, 196, 617,  29,  63,  29, 483, 137, 479, 223,  87, 198, 137,
        704,  13, 177, 363,  63, 355, 529, 130,  34, 218, 133, 379, 580,

In [ ]:
def make_loaders_for_task(task_name):
    task = safe_get_task(
        dataset_name,
        task_name,
        root_dir=root_dir,
        download=True,
    )

    train_table = safe_get_task_table(task, "train")
    val_table = safe_get_task_table(task, "val")
    test_table = safe_get_task_table(task, "test")

    loader_dict = {}

    for split_name, table in [
        ("train", train_table),
        ("val", val_table),
        ("test", test_table),
    ]:
        table_input = get_node_train_table_input(
            table=table,
            task=task,
        )

        loader_dict[split_name] = NeighborLoader(
            data=data,
            num_neighbors=[num_neighbors for _ in range(num_layers)],
            time_attr="time",
            input_nodes=table_input.nodes,
            input_time=table_input.time,
            transform=table_input.transform,
            batch_size=batch_size,
            temporal_strategy="uniform",
            shuffle=split_name == "train",
            num_workers=0,
            persistent_workers=False,
        )

    return task, train_table, val_table, test_table, loader_dict

In [ ]:
# ============================================================
# TRAIN MODELS FOR ALL REQUESTED TASKS AND SEEDS
# ============================================================

def set_all_seeds(seed):
    """
    Ensure reproducibility across Python, NumPy, and PyTorch.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# Dictionary storing all trained models and associated objects.
trained = {}

for task_name in TASK_NAMES:

    trained[task_name] = {}

    for seed in SEEDS:

        # ----------------------------------------------------
        # Reproducibility
        # ----------------------------------------------------
        set_all_seeds(seed)

        print("=" * 80)
        print("seed:", seed)
        print("=" * 80)

        # ----------------------------------------------------
        # Load task-specific data and dataloaders
        # ----------------------------------------------------
        task, train_table, val_table, test_table, loader_dict = (
            make_loaders_for_task(task_name)
        )

        entity_table = task.entity_table
        out_channels = 1

        # ----------------------------------------------------
        # Automatically infer task type
        # ----------------------------------------------------
        if task_name in ["driver-dnf", "driver-top3"]:
            task_type = "classification"

        elif task_name in ["driver-position", "qualifying-position"]:
            task_type = "regression"

        else:
            raise ValueError(
                f"Unknown task type for task_name={task_name}"
            )

        # ----------------------------------------------------
        # Dataset sanity checks
        # ----------------------------------------------------
        print("task_type:", task_type)
        print("entity_table:", entity_table)

        print("train columns:",
              train_table.df.columns.tolist())
        print("val columns:",
              val_table.df.columns.tolist())
        print("test columns:",
              test_table.df.columns.tolist())

        # Inspect one training batch
        batch = next(iter(loader_dict["train"])).to(device)

        print("batch y shape:",
              batch[entity_table].y.shape)

        if task_type == "classification":

            print(
                "batch y unique:",
                torch.unique(batch[entity_table].y.cpu())
            )

        else:

            y_cpu = (
                batch[entity_table]
                .y.detach()
                .cpu()
                .float()
            )

            print(
                "batch y stats:",
                {
                    "min": float(y_cpu.min()),
                    "max": float(y_cpu.max()),
                    "mean": float(y_cpu.mean()),
                    "std": float(y_cpu.std()),
                },
            )

        # ----------------------------------------------------
        # Build the base RelBench model
        # ----------------------------------------------------
        model = Model(
            data=data,
            col_stats_dict=col_stats_dict,
            num_layers=num_layers,
            channels=channels,
            out_channels=out_channels,
            aggr=aggr,
            norm=norm,
        ).to(device)

        # ----------------------------------------------------
        # Train the model
        # ----------------------------------------------------
        train_info = train_model(
            model=model,
            loader_dict=loader_dict,
            entity_table=entity_table,
            task=task,
            val_table=val_table,
            device=device,
            epochs=epochs,
            lr=lr,
            task_type=task_type,
        )

        # ----------------------------------------------------
        # Create an attackable version of the model
        #
        # This wrapper exposes the internal graph structure
        # and allows structural perturbations (rewiring attacks)
        # to be evaluated without modifying the original model.
        # ----------------------------------------------------
        attackable_model = build_attackable_model(
            base_model=model,
            data=data,
            channels=channels,
            aggr=aggr,
            num_layers=num_layers,
        ).to(device)

        # ----------------------------------------------------
        # Reference batch used later during attack evaluation.
        #
        # The attack itself is performed on a single batch
        # sampled from the chosen split (validation by default).
        # ----------------------------------------------------
        attack_batch = next(
            iter(loader_dict[split])
        ).to(device)

        # Copy trained weights into the attackable model
        attackable_model.load_state_dict(
            model.state_dict(),
            strict=False,
        )

        # ----------------------------------------------------
        # Store everything needed for later experiments
        # ----------------------------------------------------
        trained[task_name][seed] = {
            "task": task,
            "task_type": task_type,
            "entity_table": entity_table,
            "loader_dict": loader_dict,
            "model": model,
            "attackable_model": attackable_model,
            "batch": attack_batch,
            "train_info": train_info,
        }

        print("TRAIN INFO:", train_info)

seed: 39
Loading Database object from /home/alan/.cache/relbench/rel-f1/db...
Done in 0.05 seconds.
task_type: regression
entity_table: qualifying
train columns: ['date', 'qualifyId', 'position']
val columns: ['date', 'qualifyId', 'position']
test columns: ['date', 'qualifyId']
batch y shape: torch.Size([512])
batch y stats: {'min': 1.0, 'max': 26.0, 'mean': 12.0234375, 'std': 6.68346643447876}


[2026-06-01 17:03:14] Epoch 01 | train_loss=9.333801 | val_metrics={'r2': -0.997468113899231, 'mae': 6.966855525970459, 'rmse': 8.55636978149414}
TRAIN INFO: {'best_val_metric': 6.966855525970459, 'task_type': 'regression'}


In [ ]:
# ============================================================
# RELATIONS ELIGIBLE FOR STRUCTURAL REWIRING ATTACKS
# ============================================================
#
# Each tuple follows the format:
#
#     (foreign_key_table, foreign_key_column, primary_key_table)
#
# and represents a valid foreign-key dependency in the relational
# database. During the attack, candidate perturbations are generated
# by rewiring foreign-key values while preserving referential integrity.
#
# Example:
#
#     ("results", "f2p_driverId", "drivers")
#
# means that a row in the `results` table can be reassigned from one
# driver to another valid driver in the `drivers` table.
#
# Only relations listed here are considered when generating
# structural perturbations.
#
relations = [
    ("results", "f2p_driverId", "drivers"),
    ("results", "f2p_raceId", "races"),
    ("results", "f2p_constructorId", "constructors"),

    ("qualifying", "f2p_driverId", "drivers"),
    ("qualifying", "f2p_raceId", "races"),
    ("qualifying", "f2p_constructorId", "constructors"),

    ("standings", "f2p_driverId", "drivers"),
    ("standings", "f2p_raceId", "races"),

    ("constructor_results", "f2p_constructorId", "constructors"),
    ("constructor_results", "f2p_raceId", "races"),

    ("constructor_standings", "f2p_constructorId", "constructors"),
    ("constructor_standings", "f2p_raceId", "races"),

    ("races", "f2p_circuitId", "circuits"),
]


# ============================================================
# LOAD VALIDATION DATALOADER
# ============================================================
#
# The validation split is used as the reference split for
# attack generation and evaluation throughout the notebook.
#
# Each batch contains a sampled heterogeneous graph together
# with the target labels associated with the current task.
#
val_loader = loader_dict["val"]

## Structural Attack Evaluation Pipeline

This experiment evaluates several structural attack strategies on a trained relational GNN.

For each validation batch:

1. Compute the clean performance of the model.
2. Generate candidate rewirings using gradient-based scores.
3. Build alternative candidate rankings using different normalization schemes:
   - Raw gradients
   - Global Z-score
   - Relation-wise Min-Max
   - Robust Z-score
4. Construct additional baselines:
   - Pure random rewiring
   - Random shortlist with exact reranking
5. Apply attacks under multiple budgets.
6. Measure:
   - Task performance degradation
   - Attack diversity statistics
   - Computational cost

All results are stored in three tables:
- `rows`: attack effectiveness
- `diversity_rows`: perturbation diversity
- `time_rows`: runtime measurements

In [ ]:
# ============================================================
# MAIN EVALUATION LOOP
# ============================================================
#
# For each seed and validation batch:
#   1. Compute clean performance
#   2. Generate attack candidates
#   3. Build attack rankings
#   4. Evaluate attacks under multiple budgets
#   5. Collect effectiveness, diversity and runtime statistics
#

rows = {}
diversity_rows = {}
time_rows = {}



for seed in [SEEDS]: 

    rows[seed] = []
    diversity_rows[seed] = []
    time_rows[seed] = []


    task = trained[TASK_NAME][seed]["task"]
    entity_table = trained[TASK_NAME][seed]["entity_table"]
    loader_dict = trained[TASK_NAME][seed]["loader_dict"]
    model = trained[TASK_NAME][seed]["model"]
    attackable_model = trained[TASK_NAME][seed]["attackable_model"].to(device)
    
    for batch_id, batch in enumerate(tqdm(loader_dict[split], desc=f"{split} batches")):

        batch = batch.to(device)

        clean_metric = eval_metric_on_batch(
            model=attackable_model,
            batch=batch,
            entity_table=entity_table,
            task_type=TASK_TYPE,
            metric_name=EVAL_METRIC,
        )

        print("\n==============================")
        print(f"BATCH {batch_id}")
        print(f"task_type: {TASK_TYPE}")
        print(f"clean_{EVAL_METRIC}: {clean_metric}")
        print("==============================")

        # ============================================================
        # 1) Gradient scoring RAW
        # ============================================================

        t0 = time.time()

        global_candidates_raw = []
        all_score_infos_raw = {}

        for rel in tqdm(relations, desc=f"Batch {batch_id} | gradient raw scoring", leave=False):

            if rel not in batch.edge_index_dict:
                continue

            if int(batch.edge_index_dict[rel].size(1)) == 0:
                continue

            try:
                score_info_rel = score_targeted_rewirings_once(
                    model=attackable_model,
                    batch=batch,
                    entity_table=entity_table,
                    attacked_relation=rel,
                    max_candidates_per_dst=max_candidates_per_dst,
                    direction="increase",
                    objective_mode=GRADIENT_OBJECTIVE,
                    score_norm="raw",
                    max_topk_per_child=1,
                    verbose=False,
                    plot_grad=False,
                )

                all_score_infos_raw[rel] = score_info_rel

                relation_candidates = gradient_order_from_score_info(
                    score_info_rel,
                    topk_per_child=1,
                )

                if len(relation_candidates) == 0:
                    continue

                raw_scores = np.array(
                    [c.get("score_raw", c.get("score", 0.0)) for c in relation_candidates],
                    dtype=float,
                )

                mu = raw_scores.mean()
                sigma = raw_scores.std() + 1e-12

                for c in relation_candidates:
                    c = dict(c)

                    raw_score = float(c.get("score_raw", c.get("score", 0.0)))

                    c["relation"] = rel
                    c["attacked_relation"] = rel
                    c["reverse_relation"] = default_reverse_relation(rel)

                    c["score_raw_global"] = raw_score
                    c["score_z_global"] = float((raw_score - mu) / sigma)

                    global_candidates_raw.append(c)

            except Exception as e:
                print("FAILED RAW:", rel, repr(e))

        raw_scoring_time = time.time() - t0

        global_grad_raw_order = sorted(
            global_candidates_raw,
            key=lambda x: x["score_raw_global"],
            reverse=True,
        )

        global_grad_z_order = sorted(
            global_candidates_raw,
            key=lambda x: x["score_z_global"],
            reverse=True,
        )

        relation_minmax = {}

        for rel in set(c["relation"] for c in global_candidates_raw):
            vals = np.array(
                [
                    c["score_raw_global"]
                    for c in global_candidates_raw
                    if c["relation"] == rel
                ],
                dtype=float,
            )

            relation_minmax[rel] = {
                "min": vals.min(),
                "max": vals.max(),
            }

        eps = 1e-12

        for c in global_candidates_raw:
            rel = c["relation"]
            vmin = relation_minmax[rel]["min"]
            vmax = relation_minmax[rel]["max"]

            c["score_minmax_relation"] = (
                (c["score_raw_global"] - vmin)
                / (vmax - vmin + eps)
            )

        global_grad_minmax_order = sorted(
            global_candidates_raw,
            key=lambda x: x["score_minmax_relation"],
            reverse=True,
        )

        time_rows[seed].append({
            "task": TASK_NAME,
            "task_type": TASK_TYPE,
            "eval_metric": EVAL_METRIC,
            "batch_id": batch_id,
            "method": "gradient_raw_only",
            "phase": "gradient_scoring",
            "time_sec": raw_scoring_time,
        })

        time_rows[seed].append({
            "task": TASK_NAME,
            "task_type": TASK_TYPE,
            "eval_metric": EVAL_METRIC,
            "batch_id": batch_id,
            "method": "gradient_global_z_only",
            "phase": "gradient_scoring",
            "time_sec": raw_scoring_time,
        })

        time_rows[seed].append({
            "task": TASK_NAME,
            "task_type": TASK_TYPE,
            "eval_metric": EVAL_METRIC,
            "batch_id": batch_id,
            "method": "gradient_minmax_relation",
            "phase": "gradient_scoring",
            "time_sec": raw_scoring_time,
        })

        # ============================================================
        # 2) Gradient scoring ROBUST Z-SCORE
        # ============================================================

        t0 = time.time()

        global_candidates_robust = []

        for rel in tqdm(relations, desc=f"Batch {batch_id} | gradient robust z scoring", leave=False):

            if rel not in batch.edge_index_dict:
                continue

            if int(batch.edge_index_dict[rel].size(1)) == 0:
                continue

            try:
                score_info_rel = score_targeted_rewirings_once(
                    model=attackable_model,
                    batch=batch,
                    entity_table=entity_table,
                    attacked_relation=rel,
                    max_candidates_per_dst=max_candidates_per_dst,
                    direction="increase",
                    objective_mode=GRADIENT_OBJECTIVE,
                    score_norm="robust_zscore",
                    max_topk_per_child=1,
                    verbose=False,
                    plot_grad=False,
                )

                relation_candidates = gradient_order_from_score_info(
                    score_info_rel,
                    topk_per_child=1,
                )

                for c in relation_candidates:
                    c = dict(c)

                    c["relation"] = rel
                    c["attacked_relation"] = rel
                    c["reverse_relation"] = default_reverse_relation(rel)

                    global_candidates_robust.append(c)

            except Exception as e:
                print("FAILED ROBUST Z:", rel, repr(e))

        robust_scoring_time = time.time() - t0

        global_grad_robust_order = sorted(
            global_candidates_robust,
            key=lambda x: x.get("score", x.get("score_robust_z", -np.inf)),
            reverse=True,
        )

        time_rows[seed].append({
            "task": TASK_NAME,
            "task_type": TASK_TYPE,
            "eval_metric": EVAL_METRIC,
            "batch_id": batch_id,
            "method": "gradient_robust_z_only",
            "phase": "gradient_scoring",
            "time_sec": robust_scoring_time,
        })

        if len(global_candidates_raw) == 0:
            print("No RAW global candidates were produced.")
            continue

        if len(global_candidates_robust) == 0:
            print("No ROBUST-Z global candidates were produced.")
            continue

        # ============================================================
        # 3) Exact rerank gradient shortlist
        # ============================================================

        t0 = time.time()

        global_gradient_shortlist = select_unique_relation_child(
            global_grad_z_order,
            budget=gradient_shortlist_size,
        )

        global_exact_scored = exact_rerank_candidates_by_metric(
            model=attackable_model,
            batch=batch,
            entity_table=entity_table,
            candidates=global_gradient_shortlist,
            task_type=TASK_TYPE,
            eval_metric=EVAL_METRIC,
        )

        time_rows[seed].append({
            "task": TASK_NAME,
            "task_type": TASK_TYPE,
            "eval_metric": EVAL_METRIC,
            "batch_id": batch_id,
            "method": "gradient_exact",
            "phase": "exact_rerank",
            "time_sec": time.time() - t0,
        })

        # ============================================================
        # 4) PURE RANDOM
        # ============================================================

        random_orders = []
        random_exact_orders = []

        t0_random_pool = time.time()

        for rng_seed in range(n_random):

            random_candidates = random_multirelational_rewirings(
                batch=batch,
                relations=relations,
                B=max(
                    max(budgets),
                    random_shortlist_size,
                ),
                seed=rng_seed,
            )

            random_orders.append(random_candidates)

        time_rows[seed].append({
            "task": TASK_NAME,
            "task_type": TASK_TYPE,
            "eval_metric": EVAL_METRIC,
            "batch_id": batch_id,
            "method": "pure_random",
            "phase": "build_random_pool",
            "time_sec": time.time() - t0_random_pool,
        })

        # ============================================================
        # 4-bis) Random exact rerank
        # ============================================================

        for rng_seed, random_candidates in enumerate(random_orders):

            t0 = time.time()

            random_shortlist = random_candidates[:random_shortlist_size]

            random_exact_scored = exact_rerank_candidates_by_metric(
                model=attackable_model,
                batch=batch,
                entity_table=entity_table,
                candidates=random_shortlist,
                task_type=TASK_TYPE,
                eval_metric=EVAL_METRIC,
            )

            random_exact_orders.append(random_exact_scored)

            time_rows[seed].append({
                "task": TASK_NAME,
                "task_type": TASK_TYPE,
                "eval_metric": EVAL_METRIC,
                "batch_id": batch_id,
                "method": "random_exact",
                "phase": f"exact_rerank_seed_{rng_seed}",
                "time_sec": time.time() - t0,
            })

        # ============================================================
        # 5) Budget sweep
        # ============================================================

        for B in tqdm(budgets, desc=f"Batch {batch_id} | budget sweep", leave=False):

            B = int(B)

            method_orders = {
                "gradient_raw_only": global_grad_raw_order,
                "gradient_global_z_only": global_grad_z_order,
                "gradient_minmax_relation": global_grad_minmax_order,
                "gradient_robust_z_only": global_grad_robust_order,
                "gradient_exact": global_exact_scored,
            }

            for method_name, order in method_orders.items():

                t0 = time.time()

                selected = select_unique_relation_child(
                    order,
                    budget=B,
                )

                diversity_stats = compute_diversity_stats(
                    selected=selected,
                    all_relations=relations,
                )

                attacked_batch = apply_global_rewirings(
                    batch,
                    selected,
                )

                attacked_metric = eval_metric_on_batch(
                    model=attackable_model,
                    batch=attacked_batch,
                    entity_table=entity_table,
                    task_type=TASK_TYPE,
                    metric_name=EVAL_METRIC,
                )

                elapsed = time.time() - t0

                rows[seed].append({
                    "task": TASK_NAME,
                    "task_type": TASK_TYPE,
                    "eval_metric": EVAL_METRIC,
                    "batch_id": batch_id,
                    "budget": B,
                    "method": method_name,
                    "clean_metric": float(clean_metric),
                    "attacked_metric": float(attacked_metric),
                    "delta_metric": float(attacked_metric - clean_metric),
                    "n_selected": len(selected),
                })

                diversity_rows[seed].append({
                    "task": TASK_NAME,
                    "task_type": TASK_TYPE,
                    "eval_metric": EVAL_METRIC,
                    "batch_id": batch_id,
                    "budget": B,
                    "method": method_name,
                    "n_selected": len(selected),
                    **diversity_stats,
                })

                time_rows[seed].append({
                    "task": TASK_NAME,
                    "task_type": TASK_TYPE,
                    "eval_metric": EVAL_METRIC,
                    "batch_id": batch_id,
                    "method": method_name,
                    "phase": f"apply_eval_budget_{B}",
                    "time_sec": elapsed,
                })

            # ========================================================
            # Random global
            # ========================================================

            random_metrics = []
            random_times = []
            random_n_selected = []
            random_diversities = []

            for rng_seed, random_order in enumerate(random_orders):

                
                t0 = time.time()

                random_selected = random_order[:B]

                diversity_stats = compute_diversity_stats(
                    selected=random_selected,
                    all_relations=relations,
                )

                random_batch = apply_global_rewirings(
                    batch,
                    random_selected,
                )

                random_metric = eval_metric_on_batch(
                    model=attackable_model,
                    batch=random_batch,
                    entity_table=entity_table,
                    task_type=TASK_TYPE,
                    metric_name=EVAL_METRIC,
                )

                random_metrics.append(float(random_metric))
                random_times.append(time.time() - t0)
                random_n_selected.append(len(random_selected))
                random_diversities.append(diversity_stats)

            rows[seed].append({
                "task": TASK_NAME,
                "task_type": TASK_TYPE,
                "eval_metric": EVAL_METRIC,
                "batch_id": batch_id,
                "budget": B,
                "method": "random",
                "clean_metric": float(clean_metric),
                "attacked_metric": float(np.mean(random_metrics)),
                "attacked_metric_seed_std": float(np.std(random_metrics)),
                "delta_metric": float(np.mean(random_metrics) - clean_metric),
                "n_selected": float(np.mean(random_n_selected)),
            })

            diversity_rows[seed].append({
                "task": TASK_NAME,
                "task_type": TASK_TYPE,
                "eval_metric": EVAL_METRIC,
                "batch_id": batch_id,
                "budget": B,
                "method": "random",
                "n_selected": float(np.mean(random_n_selected)),
                "n_relations_touched": float(np.mean([d["n_relations_touched"] for d in random_diversities])),
                "relation_coverage": float(np.mean([d["relation_coverage"] for d in random_diversities])),
                "max_relation_share": float(np.mean([d["max_relation_share"] for d in random_diversities])),
                "n_unique_children": float(np.mean([d["n_unique_children"] for d in random_diversities])),
                "unique_child_ratio": float(np.mean([d["unique_child_ratio"] for d in random_diversities])),
                "relation_edit_counts": [d["relation_edit_counts"] for d in random_diversities],
            })

            time_rows[seed].append({
                "task": TASK_NAME,
                "task_type": TASK_TYPE,
                "eval_metric": EVAL_METRIC,
                "batch_id": batch_id,
                "method": "random",
                "phase": f"apply_eval_budget_{B}",
                "time_sec": float(np.sum(random_times)),
            })

            # ========================================================
            # Random shortlist + exact rerank
            # ========================================================

            random_exact_metrics = []
            random_exact_times = []
            random_exact_n_selected = []
            random_exact_diversities = []

            for rng_seed, random_exact_order in enumerate(random_exact_orders):

                t0 = time.time()

                random_exact_selected = select_unique_relation_child(
                    random_exact_order,
                    budget=B,
                )

                diversity_stats = compute_diversity_stats(
                    selected=random_exact_selected,
                    all_relations=relations,
                )

                random_exact_batch = apply_global_rewirings(
                    batch,
                    random_exact_selected,
                )

                random_exact_metric = eval_metric_on_batch(
                    model=attackable_model,
                    batch=random_exact_batch,
                    entity_table=entity_table,
                    task_type=TASK_TYPE,
                    metric_name=EVAL_METRIC,
                )

                random_exact_metrics.append(float(random_exact_metric))
                random_exact_times.append(time.time() - t0)
                random_exact_n_selected.append(len(random_exact_selected))
                random_exact_diversities.append(diversity_stats)

            rows[seed].append({
                "task": TASK_NAME,
                "task_type": TASK_TYPE,
                "eval_metric": EVAL_METRIC,
                "batch_id": batch_id,
                "budget": B,
                "method": "random_exact",
                "clean_metric": float(clean_metric),
                "attacked_metric": float(np.mean(random_exact_metrics)),
                "attacked_metric_seed_std": float(np.std(random_exact_metrics)),
                "delta_metric": float(np.mean(random_exact_metrics) - clean_metric),
                "n_selected": float(np.mean(random_exact_n_selected)),
            })

            diversity_rows[seed].append({
                "task": TASK_NAME,
                "task_type": TASK_TYPE,
                "eval_metric": EVAL_METRIC,
                "batch_id": batch_id,
                "budget": B,
                "method": "random_exact",
                "n_selected": float(np.mean(random_exact_n_selected)),
                "n_relations_touched": float(np.mean([d["n_relations_touched"] for d in random_exact_diversities])),
                "relation_coverage": float(np.mean([d["relation_coverage"] for d in random_exact_diversities])),
                "max_relation_share": float(np.mean([d["max_relation_share"] for d in random_exact_diversities])),
                "n_unique_children": float(np.mean([d["n_unique_children"] for d in random_exact_diversities])),
                "unique_child_ratio": float(np.mean([d["unique_child_ratio"] for d in random_exact_diversities])),
                "relation_edit_counts": [d["relation_edit_counts"] for d in random_exact_diversities],
            })

            time_rows[seed].append({
                "task": TASK_NAME,
                "task_type": TASK_TYPE,
                "eval_metric": EVAL_METRIC,
                "batch_id": batch_id,
                "method": "random_exact",
                "phase": f"apply_eval_budget_{B}",
                "time_sec": float(np.sum(random_exact_times)),
            })

val batches:   0%|          | 0/4 [00:00<?, ?it/s]


BATCH 0
task_type: regression
clean_mae: 6.718703269958496


tensor([[    0,     0,     0,  ..., 37552, 37552, 37552],
        [    1,   313,    29,  ...,   129,   277,    46]])


tensor([[37553, 37553, 37553,  ..., 80100, 80100, 80100],
        [    0,   286,   221,  ...,   106,   417,    75]])


tensor([[  0,   0,   0,  ..., 511, 511, 511],
        [  0, 103, 339,  ..., 162, 356, 157]])


...

tensor([[25872, 25872, 25872,  ..., 64980, 64980, 64980],
        [    0,    75,   358,  ...,   129,   364,   168]])


tensor([[    0,     0,     0,  ..., 37398, 37398, 37398],
        [    1,   503,   424,  ...,   487,    76,   240]])


tensor([[    0,     0,     0,  ..., 48586, 48586, 48586],
        [    0,   267,   169,  ...,   373,    49,    55]])


tensor([[    0,     0,     0,  ..., 48533, 48533, 48533],
        [    0,   299,   300,  ...,    28,    97,   359]])


val batches:  25%|██▌       | 1/4 [08:37<25:51, 517.30s/it]


In [10]:
# ============================================================
# BUILD DATAFRAMES
# ============================================================

def flatten_seed_rows(seed_dict):
    all_rows = []

    for seed, seed_rows in seed_dict.items():
        for r in seed_rows:
            r = dict(r)
            r["seed"] = seed
            all_rows.append(r)

    return all_rows


df_attack_batches = pd.DataFrame(flatten_seed_rows(rows))
df_diversity_batches = pd.DataFrame(flatten_seed_rows(diversity_rows))
df_times_batches = pd.DataFrame(flatten_seed_rows(time_rows))

# ============================================================
# ATTACK METRICS — mean/std over seeds + batches
# ============================================================

df_attack_mean_std = (
    df_attack_batches
    .groupby(
        ["task", "task_type", "eval_metric", "budget", "method"],
        as_index=False,
    )
    .agg(
        clean_metric_mean=("clean_metric", "mean"),
        clean_metric_std=("clean_metric", "std"),

        attacked_metric_mean=("attacked_metric", "mean"),
        attacked_metric_std=("attacked_metric", "std"),

        delta_metric_mean=("delta_metric", "mean"),
        delta_metric_std=("delta_metric", "std"),

        n_selected_mean=("n_selected", "mean"),
        n_selected_std=("n_selected", "std"),

        n_seeds=("seed", "nunique"),
        n_batches=("batch_id", "nunique"),
        n_runs=("batch_id", "count"),
    )
)


# ============================================================
# DIVERSITY METRICS — mean/std over seeds + batches
# ============================================================


df_diversity_mean_std = (
    df_diversity_batches
    .groupby(
        ["task", "task_type", "eval_metric", "budget", "method"],
        as_index=False,
    )
    .agg(
        n_selected_mean=("n_selected", "mean"),
        n_selected_std=("n_selected", "std"),

        n_relations_touched_mean=("n_relations_touched", "mean"),
        n_relations_touched_std=("n_relations_touched", "std"),

        relation_coverage_mean=("relation_coverage", "mean"),
        relation_coverage_std=("relation_coverage", "std"),

        max_relation_share_mean=("max_relation_share", "mean"),
        max_relation_share_std=("max_relation_share", "std"),

        n_unique_children_mean=("n_unique_children", "mean"),
        n_unique_children_std=("n_unique_children", "std"),

        unique_child_ratio_mean=("unique_child_ratio", "mean"),
        unique_child_ratio_std=("unique_child_ratio", "std"),

        n_seeds=("seed", "nunique"),
        n_batches=("batch_id", "nunique"),
        n_runs=("batch_id", "count"),
    )
)


# ============================================================
# TIMES — mean/std over seeds + batches
# ============================================================


df_times_by_batch_method = (
    df_times_batches
    .groupby(
        [
            "task",
            "task_type",
            "eval_metric",
            "seed",
            "batch_id",
            "method",
        ],
        as_index=False,
    )
    .agg(total_time_sec=("time_sec", "sum"))
)


df_times_mean_std = (
    df_times_by_batch_method
    .groupby(
        ["task", "task_type", "eval_metric", "method"],
        as_index=False,
    )
    .agg(
        total_time_sec_mean=("total_time_sec", "mean"),
        total_time_sec_std=("total_time_sec", "std"),

        n_seeds=("seed", "nunique"),
        n_batches=("batch_id", "nunique"),
        n_runs=("batch_id", "count"),
    )
    .sort_values(["task", "task_type", "eval_metric", "total_time_sec_mean"])
)


display(df_attack_mean_std)
display(df_diversity_mean_std)
display(df_times_mean_std)

,task,task_type,eval_metric,budget,method,clean_metric_mean,clean_metric_std,attacked_metric_mean,attacked_metric_std,delta_metric_mean,delta_metric_std,n_selected_mean,n_selected_std,n_seeds,n_batches,n_runs
0,qualifying-position,regression,mae,1,gradient_exact,6.718703,NaN,6.718709,NaN,0.000006,NaN,1.0,NaN,1,1,1
1,qualifying-position,regression,mae,1,gradient_global_z_only,6.718703,NaN,6.718709,NaN,0.000006,NaN,1.0,NaN,1,1,1
2,qualifying-position,regression,mae,1,gradient_minmax_relation,6.718703,NaN,6.718713,NaN,0.000010,NaN,1.0,NaN,1,1,1
3,qualifying-position,regression,mae,1,gradient_raw_only,6.718703,NaN,6.718713,NaN,0.000010,NaN,1.0,NaN,1,1,1
4,qualifying-position,regression,mae,1,gradient_robust_z_only,6.718703,NaN,6.718709,NaN,0.000005,NaN,1.0,NaN,1,1,1
5,qualifying-position,regression,mae,1,random,6.718703,NaN,6.718703,NaN,0.000000,NaN,1.0,NaN,1,1,1
6,qualifying-position,regression,mae,1,random_exact,6.718703,NaN,6.718706,NaN,0.000003,NaN,1.0,NaN,1,1,1
7,qualifying-position,regression,mae,125,gradient_exact,6.718703,NaN,6.718813,NaN,0.000110,NaN,50.0,NaN,1,1,1
8,qualifying-position,regression,mae,125,gradient_global_z_only,6.718703,NaN,6.718889,NaN,0.000185,NaN,125.0,NaN,1,1,1
9,qualifying-position,regression,mae,125,gradient_minmax_relation,6.718703,NaN,6.718793,NaN,0.000090,NaN,125.0,NaN,1,1,1


,task,task_type,eval_metric,budget,method,n_selected_mean,n_selected_std,n_relations_touched_mean,n_relations_touched_std,relation_coverage_mean,relation_coverage_std,max_relation_share_mean,max_relation_share_std,n_unique_children_mean,n_unique_children_std,unique_child_ratio_mean,unique_child_ratio_std,n_seeds,n_batches,n_runs
0,qualifying-position,regression,mae,1,gradient_exact,1.0,NaN,1.0,NaN,0.076923,NaN,1.000000,NaN,1.0,NaN,1.0,NaN,1,1,1
1,qualifying-position,regression,mae,1,gradient_global_z_only,1.0,NaN,1.0,NaN,0.076923,NaN,1.000000,NaN,1.0,NaN,1.0,NaN,1,1,1
2,qualifying-position,regression,mae,1,gradient_minmax_relation,1.0,NaN,1.0,NaN,0.076923,NaN,1.000000,NaN,1.0,NaN,1.0,NaN,1,1,1
3,qualifying-position,regression,mae,1,gradient_raw_only,1.0,NaN,1.0,NaN,0.076923,NaN,1.000000,NaN,1.0,NaN,1.0,NaN,1,1,1
4,qualifying-position,regression,mae,1,gradient_robust_z_only,1.0,NaN,1.0,NaN,0.076923,NaN,1.000000,NaN,1.0,NaN,1.0,NaN,1,1,1
5,qualifying-position,regression,mae,1,random,1.0,NaN,1.0,NaN,0.076923,NaN,1.000000,NaN,1.0,NaN,1.0,NaN,1,1,1
6,qualifying-position,regression,mae,1,random_exact,1.0,NaN,1.0,NaN,0.076923,NaN,1.000000,NaN,1.0,NaN,1.0,NaN,1,1,1
7,qualifying-position,regression,mae,125,gradient_exact,50.0,NaN,1.0,NaN,0.076923,NaN,1.000000,NaN,50.0,NaN,1.0,NaN,1,1,1
8,qualifying-position,regression,mae,125,gradient_global_z_only,125.0,NaN,2.0,NaN,0.153846,NaN,0.904000,NaN,125.0,NaN,1.0,NaN,1,1,1
9,qualifying-position,regression,mae,125,gradient_minmax_relation,125.0,NaN,7.0,NaN,0.538462,NaN,0.432000,NaN,125.0,NaN,1.0,NaN,1,1,1


,task,task_type,eval_metric,method,total_time_sec_mean,total_time_sec_std,n_seeds,n_batches,n_runs
5,qualifying-position,regression,mae,pure_random,0.119785,NaN,1,1,1
6,qualifying-position,regression,mae,random,18.832003,NaN,1,1,1
3,qualifying-position,regression,mae,gradient_raw_only,75.667780,NaN,1,1,1
1,qualifying-position,regression,mae,gradient_global_z_only,79.969953,NaN,1,1,1
4,qualifying-position,regression,mae,gradient_robust_z_only,82.149531,NaN,1,1,1
2,qualifying-position,regression,mae,gradient_minmax_relation,86.028179,NaN,1,1,1
7,qualifying-position,regression,mae,random_exact,137.769040,NaN,1,1,1
0,qualifying-position,regression,mae,gradient_exact,158.834177,NaN,1,1,1
